# Logistics Analysis with CrewAI (Multi-Agent Sequential Pipeline)

This notebook demonstrates a **real-world CrewAI use case** — analyzing logistics operations for pharmaceutical products and generating an optimization strategy using two specialized agents.

## What You'll Learn

- How to build a **sequential multi-agent pipeline** using `Process.sequential`
- How to use **parameterized tasks** with `{variable}` placeholders and `inputs={}` at kickoff
- How the `context` parameter chains task outputs between agents
- How `allow_delegation=False` prevents agents from delegating to each other

## Architecture

```
┌─────────────────────┐         ┌──────────────────────────┐
│  Logistics Analyst   │────────▶│  Optimization Strategist  │
│  (Analysis Task)     │ context │  (Strategy Task)          │
│  - Route efficiency  │────────▶│  - Cost reduction         │
│  - Inventory trends  │         │  - Delivery optimization  │
│  - Bottleneck ID     │         │  - Actionable roadmap     │
└─────────────────────┘         └──────────────────────────┘
```

## Key Concept: Parameterized Inputs

Instead of hardcoding values with f-strings, this example uses `{products}` placeholders in task descriptions. The actual values are injected at runtime via `crew.kickoff(inputs={'products': product_list})`. This makes the crew **reusable** for different product categories.

In [ ]:
import os
from crewai import Agent, Task, Crew, Process

# ============================================================
# 1. DEFINE AGENTS
# Each agent has a role, goal, and backstory that shape its reasoning.
# Note: {products} is a placeholder — it gets replaced at runtime
# when we call crew.kickoff(inputs={'products': ...})
# ============================================================

logistics_analyst = Agent(
    role='Logistics Analyst',
    goal='Research and analyze the current state of logistics operations for {products}',
    backstory="""You are an expert in supply chain analytics with 10 years of experience. 
    Your strength lies in identifying bottlenecks in route efficiency and 
    detecting patterns in inventory turnover. You provide data-driven insights 
    that form the foundation of strategic decisions.""",
    # verbose=True,          # Uncomment to see the agent's chain-of-thought
    allow_delegation=False   # Prevents this agent from delegating to others
)

optimization_strategist = Agent(
    role='Optimization Strategist',
    goal='Develop a comprehensive optimization strategy for {products} based on analyst insights',
    backstory="""You are a veteran operations researcher. You specialize in 
    taking complex logistics data and turning it into actionable, high-efficiency 
    strategies. You excel at cost reduction and streamlining delivery workflows.""",
    # verbose=True,
    allow_delegation=False
)

# ============================================================
# 2. DEFINE PARAMETERIZED TASKS
# Task descriptions use {products} placeholders instead of f-strings.
# This makes the crew reusable — just pass different inputs at kickoff.
# ============================================================

analysis_task = Task(
    description="""Conduct a detailed research into the current logistics operations 
    for the following products: {products}. 
    Focus specifically on current route efficiency and inventory turnover trends. 
    Identify at least three key areas where performance is lagging.""",
    expected_output="A detailed report on the current logistics state and bottlenecks.",
    agent=logistics_analyst
)

strategy_task = Task(
    description="""Review the logistics analysis provided for {products}. 
    Develop a step-by-step optimization strategy to improve delivery speed 
    and reduce inventory costs. Your strategy must be practical and data-backed.""",
    expected_output="A comprehensive optimization roadmap with specific recommendations.",
    agent=optimization_strategist,
    context=[analysis_task]  # Passes the analyst's output as input to the strategist
)

# ============================================================
# 3. BUILD THE CREW
# Process.sequential ensures tasks run one after another (analyst first,
# then strategist). The alternative is Process.hierarchical for manager-
# delegated execution.
# ============================================================

logistics_crew = Crew(
    agents=[logistics_analyst, optimization_strategist],
    tasks=[analysis_task, strategy_task],
    process=Process.sequential,
    verbose=True
)

# ============================================================
# 4. EXECUTE WITH RUNTIME PARAMETERS
# The {products} placeholder in all agent goals and task descriptions
# gets replaced with the value passed here.
# ============================================================

product_list = "Pharmaceutical supplies, cold-chain vaccines, and medical PPE"
result = logistics_crew.kickoff(inputs={'products': product_list})

print("\n\n########################")
print("## OPTIMIZATION STRATEGY ##")
print("########################\n")
print(result)

## Summary

This notebook demonstrated a **sequential multi-agent pipeline** for logistics optimization:

| Component | Purpose |
|-----------|---------|
| `Agent(allow_delegation=False)` | Prevents agents from passing work to each other |
| `Task(context=[...])` | Chains output from one task as input to the next |
| `Process.sequential` | Ensures tasks execute in order |
| `kickoff(inputs={...})` | Injects runtime parameters into `{placeholder}` fields |

**Try it yourself:** Change `product_list` to analyze different supply chains — e.g., `"Fresh produce, frozen seafood, and dairy products"`.